In [11]:
import os
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from dash import Dash, Input, Output, dcc, html


In [ ]:
os.makedirs("data", exist_ok=True)

if os.path.exists("data/f1_race_results.csv"):
    df = pd.read_csv("data/f1_race_results.csv")
else:
    BASE = "https://raw.githubusercontent.com/muharsyad/formula-one-datasets/main/"

    results  = pd.read_csv(BASE + "race_results.csv")
    races    = pd.read_csv(BASE + "races.csv")[["season", "round", "raceName", "circuitId"]]
    circuits = pd.read_csv(BASE + "circuits.csv")[["circuitId", "circuitName", "country"]]

    df = results.merge(races, on=["season", "round"], how="left") \
                .merge(circuits, on="circuitId", how="left")

    # Drop cars that never took the start - they have no meaningful grid slot
    df = df[~df["status"].isin(["Did not qualify", "Did not prequalify", "Withdrew"])]

    # Pit-lane starts are recorded as grid 0 - recode to one slot behind the last qualifier
    field_size = df.groupby(["season", "round"])["grid"].transform("max")
    df["grid"] = np.where(df["grid"] == 0, field_size + 1, df["grid"])
    df = df[df["grid"] > 0]
    df["field_size"] = field_size

    # "Classified" (given a finishing position) is not the same as "finished"
    df["position"] = pd.to_numeric(df["position"], errors="coerce")
    df["classified"] = df["positionText"].astype(str).str.isdigit()
    df["dnf"] = ~df["classified"]

    # Outcome flags and positions gained (positive = moved forward)
    df["positions_gained"] = np.where(df["classified"], df["grid"] - df["position"], np.nan)
    df["won"]    = (df["position"] == 1).fillna(False)
    df["podium"] = (df["position"] <= 3).fillna(False)
    df["top10"]  = (df["position"] <= 10).fillna(False)

    cols = ["season", "round", "raceName", "circuitName", "country", "driverName",
            "constructorName", "grid", "position", "points", "status", "field_size",
            "classified", "dnf", "positions_gained", "won", "podium", "top10"]
    df = df[cols].sort_values(["season", "round", "grid"])

    df.to_csv("data/f1_race_results.csv", index=False)

print(f"{len(df):,} race entries | {df.season.min()}-{df.season.max()} | {df.circuitName.nunique()} circuits")
print("Pole win rate, all time:", f"{df[df.grid == 1].won.mean():.1%}")

25,155 race entries | 1950-2024 | 77 circuits
Pole win rate, all time: 42.4%


In [13]:
INK   = "#0E1117"   # page background
PANEL = "#161B24"   # card background
LINE  = "#2A3140"   # gridlines and borders
TEXT  = "#E8EAED"
MUTED = "#8B94A6"
AMBER = "#F2B33D"   # accent
CYAN  = "#4EC5C1"   # secondary

SANS = "'Inter', 'Helvetica Neue', Arial, sans-serif"
MONO = "'JetBrains Mono', 'SF Mono', Consolas, monospace"

OUTCOMES = {
    "won":    ("Win", AMBER),
    "podium": ("Podium (top 3)", CYAN),
    "top10":  ("Top-10 finish", "#7C9CE8"),
}

PLOT_LAYOUT = dict(
    paper_bgcolor=PANEL,
    plot_bgcolor=PANEL,
    font=dict(family=SANS, color=TEXT, size=13),
    margin=dict(l=60, r=30, t=60, b=55),
    title=dict(font=dict(size=17, color=TEXT), x=0.01, xanchor="left"),
    xaxis=dict(gridcolor=LINE, zerolinecolor=LINE, linecolor=LINE),
    yaxis=dict(gridcolor=LINE, zerolinecolor=LINE, linecolor=LINE),
    hoverlabel=dict(bgcolor=INK, font=dict(family=MONO, color=TEXT), bordercolor=LINE),
)

def style_fig(fig, **kwargs):
    """Apply the house layout, letting per-figure settings override the defaults."""
    layout = dict(PLOT_LAYOUT)
    for key, value in kwargs.items():
        if key in ("xaxis", "yaxis") and isinstance(value, dict):
            merged = dict(layout.get(key, {}))
            merged.update(value)
            layout[key] = merged
        else:
            layout[key] = value
    fig.update_layout(**layout)
    return fig

def empty_fig(message="No races match these filters. Widen the season range."):
    """Shown instead of a misleading chart when a filter combination is too narrow."""
    fig = go.Figure()
    fig.add_annotation(text=message, showarrow=False,
                       font=dict(family=SANS, size=15, color=MUTED))
    fig.update_xaxes(visible=False)
    fig.update_yaxes(visible=False)
    return style_fig(fig, height=380)


In [14]:
# more filtering
SEASON_MIN, SEASON_MAX = int(df.season.min()), int(df.season.max())
CIRCUITS = sorted(df.circuitName.dropna().unique())

def filter_data(years, circuit, options):
    d = df[(df.season >= years[0]) & (df.season <= years[1])]
    if circuit != "ALL":
        d = d[d.circuitName == circuit]
    if "full_grids" in options:
        d = d[d.field_size >= 16]
    if "classified_only" in options:
        d = d[d.classified]
    return d

# quick check
test = filter_data([2000, 2024], "ALL", ["full_grids"])
print(len(test), "entries |", f"P1 win rate: {test[test.grid == 1].won.mean():.1%}")

10042 entries | P1 win rate: 50.8%


In [15]:
def control_block(label, hint, component):
    """A labelled control with a one-line explanation of why it matters."""
    return html.Div([
        html.Label(label, className="ctl-label"),
        html.Span(hint, className="ctl-hint"),
        component,
    ], className="ctl-block")

def stat_card(label, value_id, note_id):
    """One statistic in the answer panel. IDs let the callback fill it in."""
    return html.Div([
        html.Div(label, className="stat-label"),
        html.Div("--", id=value_id, className="stat-value"),
        html.Div("", id=note_id, className="stat-note"),
    ], className="stat-card")


In [ ]:
app = Dash(__name__, title="Grid to Flag | F1 starting position analysis")
server = app.server   # exposed for deployment

MARK_STYLE = {"color": "#FFFFFF", "fontSize": "11.5px",
              "fontFamily": "'JetBrains Mono', monospace"}
READOUT = {"fontFamily": MONO, "fontSize": "28px", "fontWeight": "700",
           "color": "#FFFFFF", "letterSpacing": "-.01em", "margin": "0 0 14px"}

app.layout = html.Div([

    # ---------------- Header: state the question ----------------
    html.Header([
        html.Div([
            html.Div("COMP 4433 · Project 2", className="eyebrow"),
            html.H1("GRID TO FLAG"),
            html.P("How much of a Formula 1 result is decided before the lights go out?",
                   className="thesis"),
            html.P([
                "Qualifying sets the order cars line up in. This dashboard measures how much "
                "that starting order determines the finishing order — across 75 seasons, at any "
                "circuit, for any starting slot you pick.",
                html.Br(),
                html.Strong("Set your filters below, then read the answer panel."),
            ], className="intro"),
        ], className="head-text"),
        html.Img(src="/assets/car.svg", className="car", alt="Formula 1 car illustration"),
    ]),

    # Step 1: controls 
    html.Section([
        html.H2("1 · Choose the races you want to study", className="step-head"),
        html.P("The app opens on 2000–2024 at every circuit, which is a good starting point. "
               "Every control below filters the same set of races, and everything further down "
               "the page updates as you change them.", className="section-note"),
        html.Div([

            control_block(
                "Seasons",
                "Drag either handle. Regulations changed enormously over 75 years, so era matters. "
                "Pick a range of several seasons — a single year won't have enough races to chart "
                "a trend.",
                html.Div([
                    html.Div(id="season-readout", style=READOUT),
                    dcc.RangeSlider(
                        id="season-range", min=SEASON_MIN, max=SEASON_MAX,
                        value=[2000, SEASON_MAX], step=1,
                        marks={y: {"label": str(y), "style": MARK_STYLE}
                               for y in range(1950, SEASON_MAX + 1, 10)},
                        className="amber-slider",
                    ),
                ]),
            ),

            html.Div([
                control_block(
                    "Circuit",
                    "Some tracks are far harder to overtake at than others. Start with all "
                    "circuits, then narrow to one — but keep the season range wide, since a "
                    "single track only hosts one race a year.",
                    dcc.Dropdown(
                        id="circuit-pick",
                        options=[{"label": "All circuits", "value": "ALL"}]
                                + [{"label": c, "value": c} for c in CIRCUITS],
                        value="ALL", clearable=False,
                    ),
                ),
                control_block(
                    "Outcome to measure",
                    "What counts as success. Wins are the sharpest test of the qualifying "
                    "advantage; top-10 finishes show how the midfield fares.",
                    dcc.RadioItems(
                        id="outcome-pick",
                        options=[{"label": v[0], "value": k} for k, v in OUTCOMES.items()],
                        value="podium", className="radio-row",
                    ),
                ),
            ], className="ctl-pair"),

            html.Div([
                control_block(
                    "Filters",
                    "Mechanical failures are not a driving outcome — drop DNFs to isolate "
                    "on-track position changes. Full grids only removes the sparse early years, "
                    "when some races had barely a dozen starters.",
                    dcc.Checklist(
                        id="filter-opts",
                        options=[
                            {"label": "Classified finishers only (drop DNFs)", "value": "classified_only"},
                            {"label": "Full grids only (16+ starters)", "value": "full_grids"},
                        ],
                        value=["full_grids"], className="check-col",
                    ),
                ),
                control_block(
                    "Your starting slot",
                    "The answer panel is calculated for this grid position. Slots near the back "
                    "have far fewer starts, so pair them with a wide season range.",
                    html.Div([
                        html.Div(id="grid-readout", style=READOUT),
                        dcc.Slider(
                            id="grid-pick", min=1, max=20, step=1, value=1,
                            marks={i: {"label": str(i), "style": MARK_STYLE}
                                   for i in [1, 5, 10, 15, 20]},
                            className="amber-slider",
                        ),
                    ]),
                ),
            ], className="ctl-pair"),

        ], className="control-panel"),
    ]),

    # Step 2: the answer 
    html.Section([
        html.H2("2 · The answer for your selection", className="step-head"),
        html.P("Rates are only shown when at least five cars started from your chosen slot. "
               "If you see a message instead of numbers, widen the seasons or move the slot "
               "forward.", className="section-note"),
        html.P(id="answer-sentence", className="answer-sentence"),
        html.Div([
            stat_card("Win rate",         "kpi-win", "kpi-win-note"),
            stat_card("Podium rate",      "kpi-pod", "kpi-pod-note"),
            stat_card("Median finish",    "kpi-med", "kpi-med-note"),
            stat_card("Failed to finish", "kpi-dnf", "kpi-dnf-note"),
        ], className="stat-row"),
    ], className="answer-block"),

    # Step 3: the evidence 
    html.Section([
        html.H2("3 · The evidence", className="step-head"),
        html.P("Each chart responds to the same controls above. Hover any point for the "
               "underlying counts. Charts that would rest on too little data say so rather "
               "than drawing a misleading line.", className="section-note"),
        html.Div([
            html.Div([
                dcc.Graph(id="fig-bar", config={"displayModeBar": False}),
                html.P("Read this as: of every car that started in slot P, this share achieved "
                       "the chosen outcome. The drop-off from the front row is the size of the "
                       "qualifying advantage. Slots with fewer than 10 starts are omitted.",
                       className="caption"),
            ], className="card"),

            html.Div([
                dcc.Graph(id="fig-heat", config={"displayModeBar": False}),
                html.P("Each column is one starting slot, normalised to 100%. A bright diagonal "
                       "means cars finish roughly where they started; a smeared column means the "
                       "race reshuffles that slot.", className="caption"),
            ], className="card"),

            html.Div([
                dcc.Graph(id="fig-line", config={"displayModeBar": False}),
                html.P("The long view: how reliably pole position converted into a win, season by "
                       "season. Rule changes, tyre wars and reliability eras all show up here. "
                       "Needs at least two seasons selected.", className="caption"),
            ], className="card card-wide"),
        ], className="chart-grid"),
    ]),

    html.Footer([
        html.P("Data: Ergast-derived Formula 1 race results, 1950–2024 (25,155 race entries). "
               "Pit-lane starts are recoded to one slot behind the last qualifier. Entries that "
               "failed to qualify are excluded, and cars that retired are counted as unclassified "
               "rather than given a finishing position.")
    ]),

], className="page")

print("layout built |", len(CIRCUITS), "circuits in dropdown")

layout built | 77 circuits in dropdown


In [17]:
@app.callback(
    Output("season-readout", "children"),
    Output("grid-readout", "children"),
    Output("answer-sentence", "children"),
    Output("kpi-win", "children"), Output("kpi-win-note", "children"),
    Output("kpi-pod", "children"), Output("kpi-pod-note", "children"),
    Output("kpi-med", "children"), Output("kpi-med-note", "children"),
    Output("kpi-dnf", "children"), Output("kpi-dnf-note", "children"),
    Input("season-range", "value"),
    Input("circuit-pick", "value"),
    Input("filter-opts", "value"),
    Input("grid-pick", "value"),
)
def update_answer(years, circuit, options, slot):
    d = filter_data(years, circuit, options)
    slot_d = d[d.grid == slot]
    where = "every circuit" if circuit == "ALL" else circuit

    season_readout = f"{years[0]} — {years[1]}"
    grid_readout   = f"P{slot}"

    # Refuse to quote a percentage from a handful of races
    if len(slot_d) < 5:
        msg = (f"Only {len(slot_d)} car(s) started P{slot} at {where} between {years[0]} and "
               f"{years[1]} — too few to quote a rate. Widen the seasons or pick another slot.")
        return (season_readout, grid_readout, msg,
                "--", "", "--", "", "--", "", "--", "")

    n   = len(slot_d)
    win = slot_d.won.mean() * 100
    pod = slot_d.podium.mean() * 100
    dnf = slot_d.dnf.mean() * 100
    med = slot_d.position.median()
    med_txt = "--" if pd.isna(med) else f"P{med:.0f}"

    sentence = (f"Starting P{slot} at {where}, {years[0]}–{years[1]}: across {n:,} race starts, "
                f"{win:.1f}% ended in victory and {pod:.1f}% ended on the podium.")

    return (
        season_readout, grid_readout, sentence,
        f"{win:.1f}%", f"{int(slot_d.won.sum()):,} wins from {n:,} starts",
        f"{pod:.1f}%", f"{int(slot_d.podium.sum()):,} podiums from {n:,} starts",
        med_txt,      "midpoint of classified finishes",
        f"{dnf:.1f}%", "retired or unclassified",
    )

print(update_answer([2000, 2024], "ALL", ["full_grids"], 15)[2])

Starting P15 at every circuit, 2000–2024: across 475 race starts, 0.2% ended in victory and 1.5% ended on the podium.


In [18]:
@app.callback(
    Output("fig-bar", "figure"),
    Output("fig-heat", "figure"),
    Output("fig-line", "figure"),
    Input("season-range", "value"),
    Input("circuit-pick", "value"),
    Input("outcome-pick", "value"),
    Input("filter-opts", "value"),
)
def update_figures(years, circuit, outcome, options):
    d = filter_data(years, circuit, options)
    label, colour = OUTCOMES[outcome]
    span  = f"{years[0]}–{years[1]}"
    where = "all circuits" if circuit == "ALL" else circuit

    if len(d) < 30:
        blank = empty_fig()
        return blank, blank, blank

    # Chart 1: bar, outcome rate by starting slot
    by_slot = (d[d.grid <= 22].groupby("grid")
               .agg(rate=(outcome, "mean"), starts=(outcome, "size"), hits=(outcome, "sum"))
               .reset_index())
    by_slot = by_slot[by_slot.starts >= 10]
    by_slot["rate"] *= 100

    fig_bar = go.Figure(go.Bar(
        x=by_slot.grid, y=by_slot.rate,
        marker=dict(color=by_slot.rate, colorscale=[[0, LINE], [1, colour]], line_width=0),
        customdata=np.stack([by_slot.hits, by_slot.starts], axis=-1),
        hovertemplate="Started <b>P%{x}</b><br>%{y:.1f}% achieved it"
                      "<br>%{customdata[0]:,} of %{customdata[1]:,} starts<extra></extra>",
    ))
    style_fig(fig_bar, height=380,
              title=f"Chance of a {label.lower()} by starting slot · {where}, {span}",
              xaxis_title="Starting grid position", yaxis_title=f"{label} rate (%)",
              showlegend=False)
    fig_bar.update_xaxes(dtick=1)
    fig_bar.update_yaxes(ticksuffix="%")

    # Chart 2: heatmap, grid vs finish
    h = d[(d.grid <= 20) & (d.position <= 20)].dropna(subset=["position"])
    if len(h) >= 50:
        mat = pd.crosstab(h.position.astype(int), h.grid.astype(int), normalize="columns") * 100
        fig_heat = px.imshow(
            mat,
            labels=dict(x="Starting grid position", y="Finishing position", color="Share"),
            color_continuous_scale=[[0, PANEL], [0.5, "#3E5A6B"], [1, CYAN]],
            aspect="auto", origin="upper",
        )
        fig_heat.update_traces(hovertemplate="Started P%{x} → finished P%{y}"
                                             "<br>%{z:.1f}% of that slot's finishers<extra></extra>")
        style_fig(fig_heat, height=430,
                  title=f"Where each grid slot actually finishes · {where}, {span}",
                  coloraxis_colorbar=dict(title="% of<br>column", ticksuffix="%"))
        fig_heat.update_xaxes(dtick=2)
        fig_heat.update_yaxes(dtick=2)
    else:
        fig_heat = empty_fig("Not enough classified finishes to build the grid-vs-finish matrix.")

    # Chart 3: line, win rate from the front by season
    pole  = d[d.grid == 1].groupby("season").agg(rate=("won", "mean"), n=("won", "size"))
    pole  = pole[pole.n >= 3].reset_index()
    front = d[d.grid <= 3].groupby("season").agg(rate=("won", "mean"), n=("won", "size"))
    front = front[front.n >= 6].reset_index()

    if len(pole) >= 2:
        fig_line = go.Figure()
        fig_line.add_trace(go.Scatter(
            x=pole.season, y=pole.rate * 100, mode="lines+markers", name="Pole sitter",
            line=dict(color=AMBER, width=2.5), marker=dict(size=6),
            hovertemplate="%{x}<br>Pole won %{y:.0f}% of races<extra></extra>"))
        fig_line.add_trace(go.Scatter(
            x=front.season, y=front.rate * 100, mode="lines", name="Top-3 starter",
            line=dict(color=CYAN, width=2, dash="dash"),
            hovertemplate="%{x}<br>Top-3 starters won %{y:.0f}% of the time<extra></extra>"))
        style_fig(fig_line, height=380,
                  title=f"Win rate from the front, season by season · {where}",
                  xaxis_title="Season", yaxis_title="Share of races won (%)",
                  legend=dict(orientation="h", y=1.02, x=1, xanchor="right",
                              yanchor="bottom", bgcolor="rgba(0,0,0,0)"))
        fig_line.update_yaxes(ticksuffix="%", rangemode="tozero")
    else:
        fig_line = empty_fig("Select at least two seasons to see the trend over time.")

    return fig_bar, fig_heat, fig_line

# # test before running the server
# figs = update_figures([2000, 2024], "ALL", "podium", ["full_grids"])
# print("chart types:", [f.data[0].type for f in figs])

In [ ]:
os.makedirs("assets", exist_ok=True)

CAR = """<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 500 172" fill="none">
  <defs>
    <linearGradient id="bodyg" x1="0" y1="0" x2="0" y2="1">
      <stop offset="0" stop-color="#EDEFF3"/><stop offset="1" stop-color="#B9C0CC"/>
    </linearGradient>
  </defs>
  <path d="M40 44 H118 L114 53 H44 Z" fill="#F2B33D"/>
  <path d="M48 58 H116 L113 65 H51 Z" fill="#C6CBD4"/>
  <rect x="30" y="40" width="9" height="54" rx="2" fill="#EDEFF3"/>
  <path d="M96 65 L106 96 L96 96 L88 66 Z" fill="#C6CBD4"/>
  <path d="M46 92 H112 L110 99 H48 Z" fill="#8B94A6"/>
  <path d="M150 122 L392 118 L392 130 L150 132 Z" fill="#8B94A6"/>
  <path d="M104 100 L152 108 L150 132 L100 126 Z" fill="#6E7789"/>
  <path d="M104 98 C 130 96, 156 92, 178 86 L 196 68 C 202 58, 212 54, 224 54
           L 238 54 C 249 54, 255 61, 256 72 L 258 88
           C 296 92, 336 98, 372 104 L 424 114
           C 436 116, 441 119, 440 124 L 439 128
           L 372 124 L 300 126 L 180 126 L 112 120 Z" fill="url(#bodyg)"/>
  <path d="M172 92 L 262 90 C 268 90, 270 94, 269 100 L 266 118 L 168 122
           C 160 122, 158 116, 160 108 Z" fill="#D5DAE2"/>
  <path d="M258 92 L268 94 L266 112 L256 110 Z" fill="#0E1117"/>
  <path d="M162 118 L266 114 L266 122 L164 126 Z" fill="#9AA3B2"/>
  <path d="M176 88 L256 86 L258 94 L174 96 Z" fill="#F2B33D"/>
  <path d="M298 101 L376 111 L374 117 L296 107 Z" fill="#F2B33D"/>
  <path d="M226 62 C 240 62, 250 70, 254 82 L 228 78 Z" fill="#0E1117"/>
  <path d="M214 58 C 236 56, 254 66, 262 84" stroke="#0E1117" stroke-width="8"
        stroke-linecap="round" fill="none"/>
  <path d="M214 58 C 236 56, 254 66, 262 84" stroke="#4EC5C1" stroke-width="3.5"
        stroke-linecap="round" fill="none"/>
  <path d="M232 60 L 236 78" stroke="#0E1117" stroke-width="5" stroke-linecap="round"/>
  <rect x="256" y="76" width="14" height="6" rx="2" fill="#8B94A6"/>
  <path d="M392 129 H472 L468 139 H390 Z" fill="#F2B33D"/>
  <path d="M404 141 H470 L467 148 H402 Z" fill="#C6CBD4"/>
  <rect x="468" y="116" width="9" height="38" rx="2" fill="#EDEFF3"/>
  <circle cx="126" cy="106" r="38" fill="#12161F" stroke="#2A3140" stroke-width="3"/>
  <circle cx="126" cy="106" r="19" fill="#1E2836" stroke="#F2B33D" stroke-width="3"/>
  <circle cx="126" cy="106" r="6" fill="#F2B33D"/>
  <path d="M88 96 H164" stroke="#F2B33D" stroke-width="2.5" opacity=".8"/>
  <circle cx="380" cy="112" r="36" fill="#12161F" stroke="#2A3140" stroke-width="3"/>
  <circle cx="380" cy="112" r="18" fill="#1E2836" stroke="#F2B33D" stroke-width="3"/>
  <circle cx="380" cy="112" r="6" fill="#F2B33D"/>
  <path d="M344 103 H416" stroke="#F2B33D" stroke-width="2.5" opacity=".8"/>
  <rect x="0" y="66" width="26" height="4" rx="2" fill="#F2B33D" opacity=".5"/>
  <rect x="4" y="78" width="16" height="4" rx="2" fill="#F2B33D" opacity=".28"/>
</svg>"""
with open("assets/car.svg", "w") as f:
    f.write(CAR)

CSS = """
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700&family=JetBrains+Mono:wght@400;600;700&display=swap');

:root {
  --ink:#0E1117; --panel:#161B24; --line:#2A3140;
  --text:#E8EAED; --muted:#9BA4B5; --amber:#F2B33D; --cyan:#4EC5C1;
  --mono:'JetBrains Mono','SF Mono',Consolas,monospace;
  --sans:'Inter','Helvetica Neue',Arial,sans-serif;
}
* { box-sizing:border-box; }
body { margin:0; background:var(--ink); color:var(--text); font-family:var(--sans);
       -webkit-font-smoothing:antialiased; }
.page { max-width:1240px; margin:0 auto; padding:44px 28px 80px; }

header { border-bottom:1px solid var(--line); padding-bottom:26px; margin-bottom:8px; }
.head-text { min-width:0; }
.car { display:block; width:100%; max-width:520px; height:auto; margin:26px auto 0; }
.eyebrow { font-family:var(--mono); font-size:11px; letter-spacing:.22em;
           text-transform:uppercase; color:var(--muted); margin-bottom:14px; }
header h1 { font-family:var(--mono); font-size:clamp(36px,6.5vw,72px); font-weight:700;
            letter-spacing:-.02em; margin:0; line-height:.95; width:fit-content; }
header h1::after { content:""; display:block; width:100%; height:5px; margin-top:16px;
  background:var(--amber); box-shadow:none; }
.thesis { font-size:clamp(17px,2.2vw,22px); font-weight:500; color:var(--amber);
          margin:22px 0 12px; max-width:62ch; }
.intro { color:var(--muted); max-width:76ch; line-height:1.65; margin:0; font-size:15px; }
.intro strong { color:var(--text); font-weight:600; }

.step-head { font-family:var(--mono); font-size:13px; font-weight:600; letter-spacing:.14em;
  text-transform:uppercase; color:var(--muted); border-top:1px solid var(--line);
  padding-top:16px; margin:30px 0 20px; }
.section-note { color:var(--muted); font-size:14px; margin:-6px 0 20px; }

.control-panel { background:var(--panel); border:1px solid var(--line); border-radius:10px;
                 padding:26px 28px 34px; }
.ctl-pair { display:grid; grid-template-columns:1fr 1fr; gap:34px; margin-top:34px; }
.ctl-block { min-width:0; }
.ctl-label { display:block; font-family:var(--mono); font-size:12px; font-weight:600;
  letter-spacing:.12em; text-transform:uppercase; color:var(--text); margin-bottom:5px; }
.ctl-hint { display:block; font-size:13px; color:var(--muted); line-height:1.5; margin-bottom:16px; }

.radio-row label, .check-col label { display:flex; align-items:center; gap:9px;
  color:var(--text); font-size:14px; margin:0 0 10px; cursor:pointer; line-height:1.35; }
.radio-row input, .check-col input { accent-color:var(--amber); width:15px; height:15px;
  margin:0; flex:none; }

.amber-slider { padding-bottom:28px!important; }
.amber-slider [class*="rail"] { background:var(--line)!important; }
.amber-slider [class*="track"] { background:var(--amber)!important; }
.amber-slider [class*="handle"] { background:var(--ink)!important;
  border:2px solid var(--amber)!important; box-shadow:none!important; opacity:1!important; }
.amber-slider [class*="dot"] { background:var(--line)!important; border-color:var(--line)!important; }

#circuit-pick, #circuit-pick * {
  background-color:var(--ink)!important; color:var(--text)!important;
  border-color:var(--line)!important; }
#circuit-pick svg { fill:var(--muted)!important; }
#circuit-pick [class*="option"]:hover,
#circuit-pick [class*="focused"] { background-color:var(--line)!important; }
#circuit-pick [class*="control"] { border-radius:7px!important; box-shadow:none!important;
  min-height:40px!important; }

.answer-sentence { font-size:clamp(16px,2vw,20px); line-height:1.55; color:var(--text);
  background:var(--panel); border-left:4px solid var(--amber); border-radius:0 8px 8px 0;
  padding:20px 24px; margin:0 0 20px; max-width:88ch; }
.stat-row { display:grid; grid-template-columns:repeat(4,1fr); gap:16px; }
.stat-card { background:var(--panel); border:1px solid var(--line); border-radius:10px;
             padding:20px 22px; }
.stat-label { font-family:var(--mono); font-size:11px; letter-spacing:.14em;
  text-transform:uppercase; color:var(--muted); margin-bottom:10px; }
.stat-value { font-family:var(--mono); font-size:36px; font-weight:700; color:var(--amber);
  line-height:1; letter-spacing:-.02em; }
.stat-note { font-size:12px; color:var(--muted); margin-top:9px; line-height:1.45; }

.chart-grid { display:grid; grid-template-columns:1fr 1fr; gap:20px; }
.card { background:var(--panel); border:1px solid var(--line); border-radius:10px;
        padding:8px 8px 4px; }
.card-wide { grid-column:1 / -1; }
.caption { color:var(--muted); font-size:13px; line-height:1.6; padding:14px 18px 16px;
  margin:0; border-top:1px solid var(--line); }

footer { margin-top:52px; padding-top:22px; border-top:1px solid var(--line);
  color:var(--muted); font-size:12.5px; line-height:1.6; }

@media (max-width:900px) {
  .page { padding:30px 18px 60px; }
  .car { margin:20px auto 0; max-width:100%; }
  .ctl-pair, .chart-grid { grid-template-columns:1fr; gap:26px; }
  .stat-row { grid-template-columns:1fr 1fr; }
}
:focus-visible { outline:2px solid var(--amber); outline-offset:2px; }
@media (prefers-reduced-motion:reduce) { * { animation:none!important; transition:none!important; } }

/* ================= slider text fixes ================= */
div[class*="rc-slider-mark"] span, span[class*="rc-slider-mark-text"],
div[class*="mark"] > span {
  color:#FFFFFF!important; opacity:1!important;
  font-family:'JetBrains Mono',monospace!important; font-size:11.5px!important; }

input[type="text"], input[type="number"], .rc-input-number-input,
div[class*="input"] input {
  color:#0E1117!important;
  -webkit-text-fill-color:#0E1117!important;
  opacity:1!important;
  font-family:'JetBrains Mono',monospace!important;
  font-weight:700!important; }
input::placeholder { color:#5A6270!important; -webkit-text-fill-color:#5A6270!important; }
"""
with open("assets/style.css", "w") as f:
    f.write(CSS)

wrote assets/style.css and assets/car.svg


In [ ]:
# created Github requirements

import nbformat

NOTEBOOK = "f1_grid.ipynb"

nb = nbformat.read(NOTEBOOK, as_version=4)

lines = ['"""',
         'Grid to Flag — How much of a Formula 1 result is decided before the lights go out?',
         'COMP 4433 · Project 2',
         '',
         'Run with:  python app.py     then open http://127.0.0.1:8050',
         '"""', ""]

for cell in nb.cells:
    if cell.cell_type != "code":
        continue
    src = cell.source

    if ("nbformat" in src or src.strip().startswith("app.run")
            or "assets/style.css" in src):
        continue

    src = "\n".join(ln for ln in src.split("\n")
                    if not ln.strip().startswith("print("))

    lines.append(src.rstrip())
    lines.append("")

lines += ['',
          'if __name__ == "__main__":',
          '    import webbrowser, threading',
          '    threading.Timer(1.5, lambda: webbrowser.open("http://127.0.0.1:8050")).start()',
          '    app.run(debug=False)',
          '']

with open("app.py", "w") as f:
    f.write("\n".join(lines))

with open("requirements.txt", "w") as f:
    f.write("dash>=2.16\nplotly>=5.20\npandas>=2.0\nnumpy>=1.24\n")

with open(".gitignore", "w") as f:
    f.write("__pycache__/\n*.pyc\n.ipynb_checkpoints/\n.venv/\nvenv/\n.DS_Store\n.vscode/\n")

wrote app.py, requirements.txt, .gitignore
